# Import Dependencies

In [1]:
# COCO related libraries
from samples.coco import coco

# MaskRCNN libraries
from mrcnn.config import Config
import mrcnn.utils as utils
from mrcnn import visualize
import mrcnn.model_203 as modellib


# Misc
import os
import sys
import json
import numpy as np
import time
from PIL import Image, ImageDraw

Using TensorFlow backend.


## Constants

In [2]:
## Number of classes in dataset. Must be of type integer
NUM_CLASSES = 5

# Relative path to .h5 weights file
#WEIGHTS_FILE = None

WEIGHTS_FILE = "logs/resnext_newdata_many_v220230907T0118/mask_rcnn_resnext_newdata_many_v2_0392.h5"

# Relative path to annotations JSON file
#TRAIN_ANNOTATIONS_FILE = "datasets/aug_1/train/3_classes_train.json"
TRAIN_ANNOTATIONS_FILE = "datasets/aug_1/train/train.json"

# Relative path to directory of images that pertain to annotations file
TRAIN_ANNOTATION_IMAGE_DIR = 'datasets/aug_1/train'

# Relative path to annotations JSON file
VALIDATION_ANNOTATIONS_FILE = "datasets/aug_1/valid/valid.json"

# Relative path to directory of images that pertain to annotations file
VALIDATION_ANNOTATION_IMAGE_DIR = 'datasets/aug_1/valid'

# Number of epochs to train dataset on
NUM_EPOCHS = 650

MODEL_NAME = "resnext_newdata_many_v2"

#MODEL_NAME = "just try"

## Additional setup

In [3]:
# Set the ROOT_DIR variable to the root directory of the Mask_RCNN git repo
ROOT_DIR = os.getcwd()

# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT_DIR, "logs")

# Select which GPU to use
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID";
os.environ["CUDA_VISIBLE_DEVICES"]="0"; 

## Declare training configuration

In [4]:
class TrainConfig(coco.CocoConfig):
    """Configuration for training where MRCNN has two mask layers
    """
    # Give the configuration a recognizable name
    NAME = MODEL_NAME

    # Train on 1 image per GPU. Batch size is 1 (GPUs * images/GPU).
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

    # Number of classes (including background)
    NUM_CLASSES = 1 + NUM_CLASSES
    
    #layers
    FPN_CLASSIF_FC_LAYERS_SIZE = 1024
    
    # Min and max image dimensions
    IMAGE_MIN_DIM = 1152
    IMAGE_MAX_DIM = 1280
 #   IMAGE_MIN_DIM = 800
#    IMAGE_MAX_DIM = 1024

    # You can experiment with this number to see if it improves training
    STEPS_PER_EPOCH = 100

    # This is how often validation is run. If you are using too much hard drive space
    # on saved models (in the MODEL_DIR), try making this value larger.
    VALIDATION_STEPS = 100
    
    # Learning rate
    LEARNING_RATE = 0.001

    # Alpha for changing mask loss weights
    ALPHA = 1.
    
    LOSS_WEIGHTS = {
        "rpn_class_loss": 1.,
        "rpn_bbox_loss": 1.,
        "mrcnn_class_loss": 1.,
        "mrcnn_bbox_loss": 1.,
        "mrcnn_mask_loss": ALPHA* 1.
    }    
    
    # Matterport originally used resnet101, but I downsized to fit it on my graphics card
    # ["resnet50", "resnet101", "resnet152", "resnet203", "resnetxt50", "resnetxt101", "resnetxt152",  "resnetxt203"]
    BACKBONE = 'resnetxt152'
    
    CARDINALITY = 32

    # To be honest, I haven't taken the time to figure out what these do
    RPN_ANCHOR_SCALES = (32, 64, 128, 256, 512)

    
    # Changed to 512 because that's how many the original MaskRCNN paper used
    TRAIN_ROIS_PER_IMAGE = 512# 200 #512
    MAX_GT_INSTANCES = 256# 114 # 256
    POST_NMS_ROIS_INFERENCE = 2000 #1000 # change 2000
    POST_NMS_ROIS_TRAINING = 2000 
    
    DETECTION_MAX_INSTANCES = 400# 114 # 400
    DETECTION_MIN_CONFIDENCE = 0.5

## Display configuration

In [5]:
TrainConfig().display()


Configurations:
ALPHA                          1.0
BACKBONE                       resnetxt152
BACKBONE_STRIDES               [4, 8, 16, 32, 64]
BATCH_SIZE                     1
BBOX_STD_DEV                   [0.1 0.1 0.2 0.2]
CARDINALITY                    32
COMPUTE_BACKBONE_SHAPE         None
DETECTION_MAX_INSTANCES        400
DETECTION_MIN_CONFIDENCE       0.5
DETECTION_NMS_THRESHOLD        0.3
FPN_CLASSIF_FC_LAYERS_SIZE     1024
GPU_COUNT                      1
GRADIENT_CLIP_NORM             5.0
IMAGES_PER_GPU                 1
IMAGE_CHANNEL_COUNT            3
IMAGE_MAX_DIM                  1280
IMAGE_META_SIZE                18
IMAGE_MIN_DIM                  1152
IMAGE_MIN_SCALE                0
IMAGE_RESIZE_MODE              square
IMAGE_SHAPE                    [1280 1280    3]
LEARNING_MOMENTUM              0.9
LEARNING_RATE                  0.001
LOSS_WEIGHTS                   {'rpn_class_loss': 1.0, 'rpn_bbox_loss': 1.0, 'mrcnn_class_loss': 1.0, 'mrcnn_bbox_loss': 1.0, 'mrcn

## Create class to load dataset

In [6]:
class CocoLikeDataset(utils.Dataset):
    """ Generates a COCO-like dataset, i.e. an image dataset annotated in the style of the COCO dataset.
        See http://cocodataset.org/#home for more information.
    """
    def load_data(self, annotation_json, images_dir):
        """ Load the coco-like dataset from json
        Args:
            annotation_json: The path to the coco annotations json file
            images_dir: The directory holding the images referred to by the json file
        """
        # Load json from file
        json_file = open(annotation_json)
        coco_json = json.load(json_file)
        json_file.close()
        
        # Add the class names using the base method from utils.Dataset
        source_name = "coco_like"
        for category in coco_json['categories']:
            class_id = category['id']
            class_name = category['name']
            if class_id < 1:
                print('Error: Class id for "{}" cannot be less than one. (0 is reserved for the background)'.format(class_name))
                return
            
            self.add_class(source_name, class_id, class_name)
        
        # Get all annotations
        annotations = {}
        for annotation in coco_json['annotations']:
            image_id = annotation['image_id']
            if image_id not in annotations:
                annotations[image_id] = []
            annotations[image_id].append(annotation)
        
        # Get all images and add them to the dataset
        seen_images = {}
        for image in coco_json['images']:
            image_id = image['id']
            if image_id in seen_images:
                print("Warning: Skipping duplicate image id: {}".format(image))
            else:
                seen_images[image_id] = image
                try:
                    image_file_name = image['file_name']
                    image_width = image['width']
                    image_height = image['height']
                except KeyError as key:
                    print("Warning: Skipping image (id: {}) with missing key: {}".format(image_id, key))
                
                image_path = os.path.abspath(os.path.join(images_dir, image_file_name))
                image_annotations = annotations[image_id]
                
                # Add the image using the base method from utils.Dataset
                self.add_image(
                    source=source_name,
                    image_id=image_id,
                    path=image_path,
                    width=image_width,
                    height=image_height,
                    annotations=image_annotations
                )
                
    def load_mask(self, image_id):
        """ Load instance masks for the given image.
        MaskRCNN expects masks in the form of a bitmap [height, width, instances].
        Args:
            image_id: The id of the image to load masks for
        Returns:
            masks: A bool array of shape [height, width, instance count] with
                one mask per instance.
            class_ids: a 1D array of class IDs of the instance masks.
        """
        image_info = self.image_info[image_id]
        annotations = image_info['annotations']
        instance_masks = []
        class_ids = []
        
        for annotation in annotations:
            class_id = annotation['category_id']
            mask = Image.new('1', (image_info['width'], image_info['height']))
            mask_draw = ImageDraw.ImageDraw(mask, '1')
            for segmentation in annotation['segmentation']:
                mask_draw.polygon(segmentation, fill=1)
                bool_array = np.array(mask) > 0
                instance_masks.append(bool_array)
                class_ids.append(class_id)

        mask = np.dstack(instance_masks)
        class_ids = np.array(class_ids, dtype=np.int32)
        
        return mask, class_ids

## Load train and validation datasets

In [7]:
dataset_train = CocoLikeDataset()
dataset_train.load_data(TRAIN_ANNOTATIONS_FILE, TRAIN_ANNOTATION_IMAGE_DIR)
dataset_train.prepare()

dataset_val = CocoLikeDataset()
dataset_val.load_data(VALIDATION_ANNOTATIONS_FILE, VALIDATION_ANNOTATION_IMAGE_DIR)
dataset_val.prepare()

## Build MaskRCNN Model

In [8]:
# Create model in training mode
model = modellib.MaskRCNN(mode = "training", config = TrainConfig(), model_dir = MODEL_DIR)

Cardinality, network, blocks:  32 resnetxt152 35


W0908 01:53:02.791050 139662999295808 deprecation.py:323] From /home/venv/lib/python3.6/site-packages/tensorflow_core/python/ops/array_ops.py:1475: where (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
W0908 01:53:02.936012 139662999295808 deprecation.py:506] From /home/mrcnn/model_203.py:693: calling crop_and_resize_v1 (from tensorflow.python.ops.image_ops_impl) with box_ind is deprecated and will be removed in a future version.
Instructions for updating:
box_ind is deprecated, use box_indices instead


## Load weights into model if weights file is not None
### This is meant to be used if you are refining on a set of preexisting weights

In [9]:
if WEIGHTS_FILE is not None:
    model.load_weights(WEIGHTS_FILE, by_name = True)

Re-starting from epoch 392


## Train model
### The model after each epoch will be saved in the logs folder

In [ ]:
start_train = time.time()
model.train(dataset_train, dataset_val, learning_rate = TrainConfig().LEARNING_RATE, epochs = NUM_EPOCHS, layers = 'all')
history = model.keras_model.history.history

end_train = time.time()
minutes = round((end_train - start_train) / 60, 2)
print(f'Training took {minutes} minutes')


Starting at epoch 392. LR=0.001

Checkpoint Path: /home/logs/resnext_newdata_many_v220230907T0118/mask_rcnn_resnext_newdata_many_v2_{epoch:04d}.h5
Selecting layers to train
conv1                  (Conv2D)
bn_conv1               (BatchNorm)
res2a_branch2a         (Conv2D)
bn2a_branch2a          (BatchNorm)
res2a_branch2b_0       (Conv2D)
res2a_branch2b_1       (Conv2D)
res2a_branch2b_2       (Conv2D)
res2a_branch2b_3       (Conv2D)
res2a_branch2b_4       (Conv2D)
res2a_branch2b_5       (Conv2D)
res2a_branch2b_6       (Conv2D)
res2a_branch2b_7       (Conv2D)
res2a_branch2b_8       (Conv2D)
res2a_branch2b_9       (Conv2D)
res2a_branch2b_10      (Conv2D)
res2a_branch2b_11      (Conv2D)
res2a_branch2b_12      (Conv2D)
res2a_branch2b_13      (Conv2D)
res2a_branch2b_14      (Conv2D)
res2a_branch2b_15      (Conv2D)
res2a_branch2b_16      (Conv2D)
res2a_branch2b_17      (Conv2D)
res2a_branch2b_18      (Conv2D)
res2a_branch2b_19      (Conv2D)
res2a_branch2b_20      (Conv2D)
res2a_branch2b_21   

/home/venv/lib/python3.6/site-packages/tensorflow_core/python/framework/indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
/home/venv/lib/python3.6/site-packages/tensorflow_core/python/framework/indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
/home/venv/lib/python3.6/site-packages/tensorflow_core/python/framework/indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
/home/venv/lib/python3.6/site-packages/keras/engine/training.py:2087: UserWarning: Using a generator with `use_multiprocessi

Epoch 393/650
 99/100 [============================>.] - ETA: 32s - loss: 1.1615 - rpn_class_loss: 0.0602 - rpn_bbox_loss: 0.2386 - mrcnn_class_loss: 0.1937 - mrcnn_bbox_loss: 0.2909 - mrcnn_mask_loss: 0.3741 

/home/venv/lib/python3.6/site-packages/keras/engine/training.py:2348: UserWarning: Using a generator with `use_multiprocessing=True` and multiple workers may duplicate your data. Please consider using the`keras.utils.Sequence class.
  UserWarning('Using a generator with `use_multiprocessing=True`'


100/100 [==============================] - 3289s 33s/step - loss: 1.1545 - rpn_class_loss: 0.0600 - rpn_bbox_loss: 0.2373 - mrcnn_class_loss: 0.1921 - mrcnn_bbox_loss: 0.2887 - mrcnn_mask_loss: 0.3724 - val_loss: 1.0862 - val_rpn_class_loss: 0.0414 - val_rpn_bbox_loss: 0.2404 - val_mrcnn_class_loss: 0.1767 - val_mrcnn_bbox_loss: 0.2847 - val_mrcnn_mask_loss: 0.3391
Epoch 394/650
100/100 [==============================] - 147s 1s/step - loss: 1.1600 - rpn_class_loss: 0.0648 - rpn_bbox_loss: 0.2248 - mrcnn_class_loss: 0.2244 - mrcnn_bbox_loss: 0.2810 - mrcnn_mask_loss: 0.3610 - val_loss: 0.9476 - val_rpn_class_loss: 0.0411 - val_rpn_bbox_loss: 0.2208 - val_mrcnn_class_loss: 0.1301 - val_mrcnn_bbox_loss: 0.2514 - val_mrcnn_mask_loss: 0.3003
Epoch 395/650
100/100 [==============================] - 146s 1s/step - loss: 1.1929 - rpn_class_loss: 0.0689 - rpn_bbox_loss: 0.2526 - mrcnn_class_loss: 0.2113 - mrcnn_bbox_loss: 0.2898 - mrcnn_mask_loss: 0.3664 - val_loss: 1.0897 - val_rpn_class_loss

In [ ]:
import os

history = model.keras_model.history.history
#MODEL_NAME = 'resnet_dropout_0_2_soft_NMS_0_5_newdata_more'
os.makedirs(MODEL_NAME)
%cd $MODEL_NAME



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt



epochs = range(1,len(next(iter(history.values())))+1)
df = pd.DataFrame(history, index=epochs)
df.to_csv(MODEL_NAME+'.csv')



In [ ]:
plt.figure(figsize=(20,10))

plt.subplot(221)
plt.plot(epochs, history["loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Valid loss")
plt.ylim([0,5])
plt.legend()
plt.subplot(222)
plt.plot(epochs, history["mrcnn_class_loss"], label="Train class loss")
plt.plot(epochs, history["val_mrcnn_class_loss"], label="Valid class loss")
plt.ylim([0,5])
plt.legend()
plt.subplot(223)
plt.plot(epochs, history["mrcnn_bbox_loss"], label="Train box loss")
plt.plot(epochs, history["val_mrcnn_bbox_loss"], label="Valid box loss")
plt.ylim([0,5])
plt.legend()
plt.subplot(224)
plt.plot(epochs, history["mrcnn_mask_loss"], label="Train mask loss")
plt.plot(epochs, history["val_mrcnn_mask_loss"], label="Valid mask loss")
plt.ylim([0,5])
plt.legend()
plt.savefig("Losses")
plt.show()    

## Include evaluation scripts in training script so that the kernel does not have to be reloaded. Eases the process of rapidly training and evaluating models

### Import dependencies

In [ ]:
# COCO related libraries
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskUtils
from samples.coco import coco
from samples.coco.coco import evaluate_coco

# Misc
import os
import skimage.io
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tnrange, tqdm_notebook
%matplotlib inline 

In [ ]:
%cd ..
%cd home

In [ ]:
# Number of classes in dataset. Must be of type integer
#NUM_CLASSES = 3

# Relative path to .h5 weights file
#WEIGHTS_FILE = None

#WEIGHTS_FILE = 'logs/5_classes_resnetxt_152_epc_200_step_per_ep_220_val_steps_200_alldataset_aug20221012T0013/mask_rcnn_5_classes_resnetxt_152_epc_200_step_per_ep_220_val_steps_200_alldataset_aug_0200.h5'

# Relative path to ground truth annotations JSON filecccccccccccccccccccccccccccccccccc
TEST_ANNOTATIONS_FILE = 'datasets/aug_1/test/test.json'

# Relative path to images associated with ground truth JSON fileccccccccccccc
TEST_DATASET_DIR = 'datasets/aug_1/test/'

# Relative path to the directory of images that you want to run inferencing on
TEST_IMAGE_DIR = 'datasets/no_aug/test/'

#MODEL_NAME = "5_classes_resnetxt_152_epc_200_step_per_ep_220_val_steps_200_alldataset_aug20221012T0013"

#MODEL_NAME = "152_testv2_model"

## Declare evaluation configuration

In [ ]:
class EvalConfig(coco.CocoConfig):
    """ Configuration for evaluation """
    
    # Give the configuration a recognizable name
    NAME = MODEL_NAME
    
    # How many GPUs
    GPU_COUNT = 1
    
    # How many images per gpu
    IMAGES_PER_GPU = 1

    # Number of classes (including background)
    NUM_CLASSES = 1 + NUM_CLASSES  # background + other classes
    
    #layers
    FPN_CLASSIF_FC_LAYERS_SIZE = 1024
    
    IMAGE_MIN_DIM = 1152
    IMAGE_MAX_DIM = 1280
    
    # Alpha for changing mask loss weights
    ALPHA =1.
    
    LOSS_WEIGHTS = {
        "rpn_class_loss": 1.,
        "rpn_bbox_loss": 1.,
        "mrcnn_class_loss": 1.,
        "mrcnn_bbox_loss": 1.,
        "mrcnn_mask_loss": ALPHA* 1.
    }    
    
    # Matterport originally used resnet101, but I downsized to fit it on my graphics card
    # ["resnet50", "resnet101", "resnet152", "resnet203", "resnetxt50", "resnetxt101", "resnetxt152",  "resnetxt203"]
    BACKBONE = 'resnetxt152'
    
    CARDINALITY = 32



    # To be honest, I haven't taken the time to figure out what these do
    RPN_ANCHOR_SCALES = (32, 64, 128, 256, 512)
    
    # Changed to 512 because that's how many the original MaskRCNN paper used
    TRAIN_ROIS_PER_IMAGE = 512#200
    MAX_GT_INSTANCES = 256#114
    POST_NMS_ROIS_INFERENCE = 2000#1000 
    POST_NMS_ROIS_TRAINING = 2000 
    
    DETECTION_MAX_INSTANCES = 400#114
    DETECTION_MIN_CONFIDENCE = 0.5

## Display configuration

In [ ]:
EvalConfig().display()

## Build class to load ground truth data

In [ ]:
class CocoDataset(utils.Dataset):
    def load_coco_gt(self, annotations_file, dataset_dir):
        """Load a COCO styled ground truth dataset
        """
        
        # Create COCO object
        coco = COCO(annotations_file)

        # Load all classes
        class_ids = sorted(coco.getCatIds())

        # Load all images
        image_ids = list(coco.imgs.keys())

        # Add classes
        for i in class_ids:
            self.add_class("coco", i, coco.loadCats(i)[0]["name"])

        # Add images
        for i in image_ids:
            self.add_image(
                "coco", image_id = i,
                path = os.path.join(dataset_dir, coco.imgs[i]['file_name']),
                width = coco.imgs[i]["width"],
                height = coco.imgs[i]["height"],
                annotations = coco.loadAnns(coco.getAnnIds(
                    imgIds = [i], catIds = class_ids, iscrowd=None)))
        
        return coco
    
    def load_mask(self, image_id):
        """Load instance masks for the given image.
        Different datasets use different ways to store masks. This
        function converts the different mask format to one format
        in the form of a bitmap [height, width, instances].
        Returns:
        masks: A bool array of shape [height, width, instance count] with
            one mask per instance.
        class_ids: a 1D array of class IDs of the instance masks.
        """
        # If not a COCO image, delegate to parent class.
        image_info = self.image_info[image_id]
        if image_info["source"] != "coco":
            return super(CocoDataset, self).load_mask(image_id)

        instance_masks = []
        class_ids = []
        annotations = self.image_info[image_id]["annotations"]
        # Build mask of shape [height, width, instance_count] and list
        # of class IDs that correspond to each channel of the mask.
        for annotation in annotations:
            class_id = self.map_source_class_id(
                "coco.{}".format(annotation['category_id']))
            if class_id:
                m = self.annToMask(annotation, image_info["height"],
                                   image_info["width"])
                # Some objects are so small that they're less than 1 pixel area
                # and end up rounded out. Skip those objects.
                if m.max() < 1:
                    continue
                # Is it a crowd? If so, use a negative class ID.
                if annotation['iscrowd']:
                    # Use negative class ID for crowds
                    class_id *= -1
                    # For crowd masks, annToMask() sometimes returns a mask
                    # smaller than the given dimensions. If so, resize it.
                    if m.shape[0] != image_info["height"] or m.shape[1] != image_info["width"]:
                        m = np.ones([image_info["height"], image_info["width"]], dtype=bool)
                instance_masks.append(m)
                class_ids.append(class_id)

        # Pack instance masks into an array
        if class_ids:
            mask = np.stack(instance_masks, axis=2).astype(np.bool)
            class_ids = np.array(class_ids, dtype=np.int32)
            return mask, class_ids
        else:
            # Call super class to return an empty mask
            return super(CocoDataset, self).load_mask(image_id)

    def image_reference(self, image_id):
        """Return a link to the image in the COCO Website."""
        info = self.image_info[image_id]
        if info["source"] == "coco":
            return "http://cocodataset.org/#explore?id={}".format(info["id"])
        else:
            super(CocoDataset, self).image_reference(image_id)

    # The following two functions are from pycocotools with a few changes.

    def annToRLE(self, ann, height, width):
        """
        Convert annotation which can be polygons, uncompressed RLE to RLE.
        :return: binary mask (numpy 2D array)
        """
        segm = ann['segmentation']
        if isinstance(segm, list):
            # polygon -- a single object might consist of multiple parts
            # we merge all parts into one mask rle code
            rles = maskUtils.frPyObjects(segm, height, width)
            rle = maskUtils.merge(rles)
        elif isinstance(segm['counts'], list):
            # uncompressed RLE
            rle = maskUtils.frPyObjects(segm, height, width)
        else:
            # rle
            rle = ann['segmentation']
        return rle

    def annToMask(self, ann, height, width):
        """
        Convert annotation which can be polygons, uncompressed RLE, or RLE to binary mask.
        :return: binary mask (numpy 2D array)
        """
        rle = self.annToRLE(ann, height, width)
        m = maskUtils.decode(rle)
        return m

## Build MaskRCNN Model

In [ ]:
# Create the model in inference mode
model = modellib.MaskRCNN(mode = "inference", config = EvalConfig(), model_dir = MODEL_DIR)

## Load weights from last trained model

In [ ]:
if WEIGHTS_FILE is None:
    model.load_weights(model.find_last(), by_name = True)
else:
    model.load_weights(WEIGHTS_FILE, by_name = True)

## Load dataset

In [ ]:
dataset_val = CocoDataset()
coco = dataset_val.load_coco_gt(annotations_file = TEST_ANNOTATIONS_FILE, dataset_dir = TEST_DATASET_DIR)
dataset_val.prepare()
class_names = dataset_val.class_names

In [ ]:
print(class_names)

## Evaluate model with COCO test
### If your results come back as a bunch of zeros, check to make sure that the "width" and "height" tag in your COCO dataset are correct

In [ ]:
#evaluate_coco(model, dataset_val, coco, "segm")

## Calculating mAP as per example in train_shapes.ipynb

In [ ]:
# Compute VOC-Style mAP @ IoU=0.5
# Running on 10 images. Increase for better accuracy.
image_ids = np.random.choice(dataset_val.image_ids, len(dataset_val.image_ids))

# Instanciate arrays to create our metrics
APs = []
ARs = []
precisions_arr = []
recalls_arr = []
overlaps_arr = []
class_ids_arr = []
scores_arr = []
F1_scores = []; 

for id in tnrange(len(image_ids), desc = "Processing images in dataset..."):
    # Load image and ground truth data
    image, image_meta, gt_class_id, gt_bbox, gt_mask =\
        modellib.load_image_gt(dataset_val, EvalConfig(),
                               image_ids[id], use_mini_mask=False)
    molded_images = np.expand_dims(modellib.mold_image(image, EvalConfig()), 0)
    # Run object detection
    results = model.detect([image], verbose=0)
    r = results[0]
    # Compute AP
    AP, precisions, recalls, overlaps =\
        utils.compute_ap(gt_bbox, gt_class_id, gt_mask,
                         r["rois"], r["class_ids"], r["scores"], r['masks'])
    
    AR, positive_ids = utils.compute_recall(r["rois"], gt_bbox, iou=0.2)
    # Append AP to AP array
    APs.append(AP)
    ARs.append(AR)
    
    #F1_scores.append((2* (mean(precisions) * mean(recalls)))/(mean(precisions) + mean(recalls)))
    np.mean(APs)
    np.mean(ARs)
    
    # Append precisions
    for precision in precisions:
        precisions_arr.append(precision)
    
    # Append recalls
    for recall in recalls:
        recalls_arr.append(recall)
        
    
    
    # Append overlaps
    for overlap in overlaps:
        overlaps_arr.append(overlap)
    
    # Append class_ids
    for class_id in r["class_ids"]:
        class_ids_arr.append(class_id)
    
    # Append scores 
    for score in r["scores"]:
        scores_arr.append(score)
        
        
    
mAP =  np.mean(APs)
mAR = np.mean(ARs)
print("mAP: ", mAP)
print("mAR: ", mAR)


F1_score_2 = (2 * mAP * mAR)/(mAP + mAR)
print('second way calculate f1-score_2: ', F1_score_2)

In [ ]:
#Save the variables in a txt file 
file = open(MODEL_NAME+"/"+"variable.txt", "w")
file.write("mAP = " + str(mAP) + "\n" +"mAR = "+ str(mAR) + "\n"+"F1 = "+ str(F1_score_2) )
file.close()

## Plot precision recall curve

In [ ]:
visualize.plot_precision_recall(MODEL_NAME, AP, precisions, recalls)

# Confusion Matrix

In [ ]:
# an example of plotting confusion matrix.
# the first step consists of computing ground-truth and prediction vectors for all images.
# using these vectors, the plot_confusion_matrix_from_data function plots the CM and computes tps fps and fns
import pandas as pd
import numpy as np
import os 

#ground-truth and predictions lists
gt_tot = np.array([])
pred_tot = np.array([])
#mAP list
mAP_ = []


for id in tnrange(len(image_ids), desc = "Processing images in dataset..."):
    # Load image and ground truth data
    image, image_meta, gt_class_id, gt_bbox, gt_mask =\
        modellib.load_image_gt(dataset_val, EvalConfig(),
                               image_ids[id], use_mini_mask=False)

    # Run the model
    results = model.detect([image], verbose=1)
    r = results[0]
    
    #compute gt_tot and pred_tot
    gt, pred = utils.gt_pred_lists(gt_class_id, gt_bbox, r['class_ids'], r['rois'])
    gt_tot = np.append(gt_tot, gt)
    pred_tot = np.append(pred_tot, pred)
    
    #precision_, recall_, AP_ 
    AP_, precision_, recall_, overlap_ = utils.compute_ap(gt_bbox, gt_class_id, gt_mask,
                                          r['rois'], r['class_ids'], r['scores'], r['masks'])
    #check if the vectors len are equal
    print("the actual len of the gt vect is : ", len(gt_tot))
    print("the actual len of the pred vect is : ", len(pred_tot))
    
    mAP_.append(AP_)
    print("Average precision of this image : ",AP_)
    print("The actual mean average precision for the whole images (matterport methode) ", sum(mAP_)/len(mAP_))
   # print("Ground truth object : "+dataset_val.class_names[gt])
   # print("Predicted object : "+dataset_val.class_names[pred])


#print("ground truth list : ",gt_tot)
#print("predicted list : ",pred_tot)

tp,fp,fn, dm =utils.plot_confusion_matrix_from_data(MODEL_NAME, class_names, gt_tot,pred_tot,fz=18, figsize=(20,20), lw=0.5)

In [ ]:
image_ids

In [ ]:
# https://vitalflux.com/ml-metrics-sensitivity-vs-specificity-difference/

#Mathematically, sensitivity or true positive rate can be calculated as the following:

#Sensitivity = (True Positive)/(True Positive + False Negative)

Sensitivity =  tp/(tp+fn)


# Mathematically, specificity can be calculated as the following:

# Specificity = (True Negative)/(True Negative + False Positive)
# 

In [ ]:
print("tp for each class :",tp)
print("fp for each class :",fp)
print("fn for each class :",fn)

#eliminate the background class (class A) from tps fns and fns lists since it doesn't concern us anymore : 
del tp[0]
del fp[0]
del fn[0]


print("\n########################\n")
print("tp for each class :",tp)
print("fp for each class :",fp)
print("fn for each class :",fn)

In [ ]:
%cd ..
%cd home

In [ ]:

file_names = next(os.walk(TEST_IMAGE_DIR))[2]
image = skimage.io.imread(os.path.join(TEST_IMAGE_DIR, random.choice(file_names)))

#image = skimage.io.imread(os.path.join(TEST_IMAGE_DIR,"cell_2_5600_1184_6100_1684_jpg.rf.008a0e7fe4d7df8bba2675d172a2da69.jpg"))

# Run the model
results = model.detect([image], verbose=1)
r = results[0]

for i in range(0, len(r['class_ids'])):
    if r['class_ids'][i] == 1:
        Xm = (r['rois'][i][1] + r['rois'][i][3])/2
        Ym = (r['rois'][i][0]+ r['rois'][i][2])/2
        coord = [Xm,Ym]
        print('A cancerigenous cell was found in X, Y: ', coord)
        
    if r['class_ids'][i] == 2:
        Xm = (r['rois'][i][1] + r['rois'][i][3])/2
        Ym = (r['rois'][i][0]+ r['rois'][i][2])/2
        coord = [Xm,Ym]
        print('A dangerous cell was found in X, Y: ', coord)        


visualize.display_instances(image, r['rois'], r['masks'], r['class_ids'], class_names, r['scores'])

In [ ]:
# Load a random image from the images folder
file_names = next(os.walk(TEST_IMAGE_DIR))[2]
image = skimage.io.imread(os.path.join(TEST_IMAGE_DIR, random.choice(file_names)))

#image = skimage.io.imread("cell_2_4400_0_4900_500_jpg.rf.634fc6434c65b180a5c3a9bc993e1241.jpg")

# Run detection
results = model.detect([image], verbose=1)

# Visualize results
r = results[0]
visualize.display_instances(image, r['rois'], r['masks'], r['class_ids'], class_names, r['scores'])
